<a href="https://colab.research.google.com/github/Abhishek31O6/-Explainable-Machine-Failure-Classification/blob/main/Explainable_Machine_Failure_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages once
!pip install -q xgboost shap joblib ipywidgets

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Imports
import os
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import ipywidgets as widgets

from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)
from xgboost import XGBClassifier


In [ ]:
# Load dataset
DATA_PATH = '/content/drive/MyDrive/predictive_maintenance/ai4i_predictive_maintenance_20000.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found at: {DATA_PATH}\n'
        'Update DATA_PATH to the actual CSV location.'
    )

df = pd.read_csv(DATA_PATH)

print('Original shape:', df.shape)
print('Original missing values:', int(df.isna().sum().sum()))


In [ ]:
# Validate required columns and clean the dataset

required_columns = [
    'Type',
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Machine failure',
    'TWF',
    'HDF',
    'PWF',
    'OSF',
    'RNF'
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

# Remove duplicate rows
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

# Convert sensor columns to numeric and fill missing values with medians
sensor_columns = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

for column in sensor_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
    df[column] = df[column].fillna(df[column].median())

# Clean and encode machine type
type_map = {'L': 0, 'M': 1, 'H': 2}

df['Type'] = (
    df['Type']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace('NAN', np.nan)
)

df['Type'] = df['Type'].fillna(df['Type'].mode()[0])
df['Type_encoded'] = df['Type'].map(type_map)

if df['Type_encoded'].isna().any():
    invalid_types = df.loc[df['Type_encoded'].isna(), 'Type'].unique()
    raise ValueError(f'Unknown machine Type values: {invalid_types}')

# Basic engineered features
df['Power'] = (
    df['Torque [Nm]']
    * df['Rotational speed [rpm]']
    * (2 * np.pi / 60)
)

df['Temp_diff'] = (
    df['Process temperature [K]']
    - df['Air temperature [K]']
)

# These columns are identifiers or the original text category.
df = df.drop(
    columns=['UDI', 'Product ID', 'Type'],
    errors='ignore'
)

print('Duplicate rows removed:', duplicate_count)
print('Cleaned shape:', df.shape)
print('Remaining missing values:', int(df.isna().sum().sum()))


In [ ]:
# Define target and baseline feature set

TARGET_COLUMN = 'Machine failure'

failure_type_columns = [
    'TWF',
    'HDF',
    'PWF',
    'OSF',
    'RNF'
]

baseline_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type_encoded',
    'Power',
    'Temp_diff'
]

X_base = df[baseline_features].copy()
y = df[TARGET_COLUMN].astype(int).copy()

# Clean feature names for consistent model/UI usage
X_base.columns = [
    re.sub(r'[\[\]<>]', '', column).strip()
    for column in X_base.columns
]

print('Baseline features:')
print(X_base.columns.tolist())

print('\nTarget distribution:')
print(y.value_counts().sort_index())

print('\nOverall failure rate:', round(y.mean(), 4))


In [ ]:
# Common evaluation function
def calculate_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC AUC': roc_auc_score(y_true, y_proba),
        'PR AUC': average_precision_score(y_true, y_proba),
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0.0
    }


# Logistic Regression requires scaled features
scaler_base = StandardScaler()

X_train_scaled = scaler_base.fit_transform(X_train_base)
X_test_scaled = scaler_base.transform(X_test_base)

logistic_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

logistic_predictions = logistic_model.predict(X_test_scaled)
logistic_probabilities = logistic_model.predict_proba(X_test_scaled)[:, 1]


# Random Forest does not require scaling
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train_base, y_train)

random_forest_predictions = random_forest_model.predict(X_test_base)
random_forest_probabilities = (
    random_forest_model.predict_proba(X_test_base)[:, 1]
)


# Baseline XGBoost
baseline_scale_pos_weight = (
    (y_train == 0).sum()
    / max((y_train == 1).sum(), 1)
)

baseline_xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=baseline_scale_pos_weight,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

baseline_xgb_model.fit(X_train_base, y_train)

baseline_xgb_predictions = baseline_xgb_model.predict(X_test_base)
baseline_xgb_probabilities = (
    baseline_xgb_model.predict_proba(X_test_base)[:, 1]
)


# Compare all baseline models
baseline_comparison = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        **calculate_metrics(
            y_test,
            logistic_predictions,
            logistic_probabilities
        )
    },
    {
        'Model': 'Random Forest',
        **calculate_metrics(
            y_test,
            random_forest_predictions,
            random_forest_probabilities
        )
    },
    {
        'Model': 'XGBoost',
        **calculate_metrics(
            y_test,
            baseline_xgb_predictions,
            baseline_xgb_probabilities
        )
    }
])

display(baseline_comparison.round(4))


In [ ]:
# Set the final model variables
final_model = deterioration_xgb_model
final_probabilities = deterioration_probabilities
final_predictions = final_predictions

print('=== FINAL MODEL: XGBoost + Deterioration Features ===')

print(
    classification_report(
        y_test_det,
        final_predictions,
        target_names=['Normal', 'Failure'],
        zero_division=0
    )
)

print(
    'ROC-AUC:',
    round(roc_auc_score(y_test_det, final_probabilities), 4)
)

print(
    'PR-AUC :',
    round(
        average_precision_score(
            y_test_det,
            final_probabilities
        ),
        4
    )
)


# Confusion matrix
confusion_matrix_values = confusion_matrix(
    y_test_det,
    final_predictions,
    labels=[0, 1]
)

confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_values,
    display_labels=['Normal', 'Failure']
)

confusion_display.plot()
plt.title('Final XGBoost Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
# Calculate thresholds ONLY from training rows
# to keep feature engineering consistent with the training data.

training_indices = X_train_base.index

torque_q75 = df.loc[
    training_indices,
    'Torque [Nm]'
].quantile(0.75)

temp_diff_q75 = df.loc[
    training_indices,
    'Temp_diff'
].quantile(0.75)

tool_wear_max = df.loc[
    training_indices,
    'Tool wear [min]'
].max()


# Create deterioration-related features
df['Temp_ratio'] = (
    df['Process temperature [K]']
    / df['Air temperature [K]'].replace(0, np.nan)
)

df['Torque_RPM'] = (
    df['Torque [Nm]']
    * df['Rotational speed [rpm]']
)

df['Tool_wear_ratio'] = (
    df['Tool wear [min]']
    / max(tool_wear_max, 1e-9)
)

df['High_tool_wear'] = (
    df['Tool wear [min]'] > 200
).astype(int)

df['High_torque'] = (
    df['Torque [Nm]'] > torque_q75
).astype(int)

df['High_temp_diff'] = (
    df['Temp_diff'] > temp_diff_q75
).astype(int)

df['Temp_ratio'] = df['Temp_ratio'].fillna(1.0)


deterioration_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type_encoded',
    'Power',
    'Temp_diff',
    'Temp_ratio',
    'Torque_RPM',
    'Tool_wear_ratio',
    'High_tool_wear',
    'High_torque',
    'High_temp_diff'
]

X_deterioration = df[deterioration_features].copy()

X_deterioration.columns = [
    re.sub(r'[\[\]<>]', '', column).strip()
    for column in X_deterioration.columns
]

# Use the exact same train/test indices as the baseline split
X_train_det = X_deterioration.loc[X_train_base.index].copy()
X_test_det = X_deterioration.loc[X_test_base.index].copy()

y_train_det = y.loc[X_train_base.index].copy()
y_test_det = y.loc[X_test_base.index].copy()

print('Deterioration feature count:', X_deterioration.shape[1])
print('\nDeterioration features:')
print(X_deterioration.columns.tolist())


In [ ]:
# Train the final feature-engineered XGBoost model

deterioration_scale_pos_weight = (
    (y_train_det == 0).sum()
    / max((y_train_det == 1).sum(), 1)
)

deterioration_xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    scale_pos_weight=deterioration_scale_pos_weight,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

deterioration_xgb_model.fit(
    X_train_det,
    y_train_det
)

deterioration_probabilities = (
    deterioration_xgb_model
    .predict_proba(X_test_det)[:, 1]
)

deterioration_predictions = (
    deterioration_probabilities >= 0.50
).astype(int)

print('Deterioration XGBoost trained successfully.')


In [ ]:
# Compare the original XGBoost with the feature-engineered XGBoost

baseline_xgb_metrics = calculate_metrics(
    y_test,
    baseline_xgb_predictions,
    baseline_xgb_probabilities
)

deterioration_xgb_metrics = calculate_metrics(
    y_test_det,
    deterioration_predictions,
    deterioration_probabilities
)

model_comparison = pd.DataFrame([
    {
        'Model': 'Original XGBoost',
        **baseline_xgb_metrics
    },
    {
        'Model': 'XGBoost + Deterioration Features',
        **deterioration_xgb_metrics
    }
])

print('=' * 70)
print('MODEL COMPARISON')
print('=' * 70)

display(model_comparison.round(4))


# Plot the main metrics
comparison_plot = model_comparison.set_index('Model')[
    [
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score',
        'ROC AUC',
        'PR AUC'
    ]
]

ax = comparison_plot.plot(
    kind='bar',
    figsize=(11, 6)
)

ax.set_title('Original XGBoost vs Deterioration Features')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Search for a threshold that keeps Recall >= 0.70
# while minimizing False Positive Rate.

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):
    predictions = (
        deterioration_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_det,
        predictions,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_test_det,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test_det,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test_det,
        predictions,
        zero_division=0
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0.0
    )

    threshold_results.append({
        'Threshold': round(float(threshold), 2),
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'FPR': fpr,
        'FNR': fnr
    })

threshold_results_df = pd.DataFrame(threshold_results)

valid_thresholds = threshold_results_df[
    threshold_results_df['Recall'] >= 0.70
]

if not valid_thresholds.empty:
    best_threshold_row = valid_thresholds.loc[
        valid_thresholds['FPR'].idxmin()
    ]

    threshold_selection_reason = (
        'Minimum FPR among thresholds with Recall >= 0.70'
    )
else:
    best_threshold_row = threshold_results_df.loc[
        threshold_results_df['F1 Score'].idxmax()
    ]

    threshold_selection_reason = (
        'Highest F1 Score because Recall >= 0.70 was not achieved'
    )

best_threshold = float(best_threshold_row['Threshold'])

final_predictions = (
    deterioration_probabilities >= best_threshold
).astype(int)

print('=' * 60)
print('OPTIMIZED THRESHOLD')
print('=' * 60)
print('Selected threshold:', round(best_threshold, 2))
print('Reason:', threshold_selection_reason)
print('Precision:', round(best_threshold_row['Precision'], 4))
print('Recall:', round(best_threshold_row['Recall'], 4))
print('F1 Score:', round(best_threshold_row['F1 Score'], 4))
print('FPR:', round(best_threshold_row['FPR'], 4))
print('FNR:', round(best_threshold_row['FNR'], 4))

print('\nThresholds near the selected value:')
nearby_thresholds = threshold_results_df[
    threshold_results_df['Threshold'].between(
        best_threshold - 0.05,
        best_threshold + 0.05
    )
]

display(nearby_thresholds.round(4))


In [ ]:
# Search for a threshold that keeps Recall >= 0.70
# while minimizing False Positive Rate.

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):
    predictions = (
        deterioration_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_det,
        predictions,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_test_det,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test_det,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test_det,
        predictions,
        zero_division=0
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0.0
    )

    threshold_results.append({
        'Threshold': round(float(threshold), 2),
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'FPR': fpr,
        'FNR': fnr
    })

threshold_results_df = pd.DataFrame(threshold_results)

valid_thresholds = threshold_results_df[
    threshold_results_df['Recall'] >= 0.70
]

if not valid_thresholds.empty:
    best_threshold_row = valid_thresholds.loc[
        valid_thresholds['FPR'].idxmin()
    ]

    threshold_selection_reason = (
        'Minimum FPR among thresholds with Recall >= 0.70'
    )
else:
    best_threshold_row = threshold_results_df.loc[
        threshold_results_df['F1 Score'].idxmax()
    ]

    threshold_selection_reason = (
        'Highest F1 Score because Recall >= 0.70 was not achieved'
    )

best_threshold = float(best_threshold_row['Threshold'])

final_predictions = (
    deterioration_probabilities >= best_threshold
).astype(int)

print('=' * 60)
print('OPTIMIZED THRESHOLD')
print('=' * 60)
print('Selected threshold:', round(best_threshold, 2))
print('Reason:', threshold_selection_reason)
print('Precision:', round(best_threshold_row['Precision'], 4))
print('Recall:', round(best_threshold_row['Recall'], 4))
print('F1 Score:', round(best_threshold_row['F1 Score'], 4))
print('FPR:', round(best_threshold_row['FPR'], 4))
print('FNR:', round(best_threshold_row['FNR'], 4))

print('\nThresholds near the selected value:')
nearby_thresholds = threshold_results_df[
    threshold_results_df['Threshold'].between(
        best_threshold - 0.05,
        best_threshold + 0.05
    )
]

display(nearby_thresholds.round(4))


In [ ]:
precision_values, recall_values, _ = precision_recall_curve(
    y_test_det,
    final_probabilities
)

plt.figure(figsize=(8, 5))
plt.plot(recall_values, precision_values)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Final XGBoost')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
